# Fitbit Sleep Data Analysis

This notebook documents the cleaning and transformation of Fitbit sleep data using Python and pandas. The resulting datasets support the sleep-related findings and visualizations in the Bellabeat Smart Device Usage Analysis capstone.

In [1]:
import pandas as pd

## Load Sleep Data

The Fitbit sleep data is provided in two CSV exports containing minute-level sleep records. Both files are loaded into pandas DataFrames for inspection, cleaning, and consolidation.

In [2]:
sleep3 = pd.read_csv('data/minuteSleep3_merged.csv')

In [3]:
sleep4 = pd.read_csv('data/minuteSleep4_merged.csv')

## Inspect Data Quality

Check each sleep export for the number of records, unique participants, missing values in key fields, and duplicate rows before combining the datasets.

In [4]:
for name, data in [("Sleep Export 3", sleep3), ("Sleep Export 4", sleep4)]:
    print(f"\n{name}")
    print("Rows:", len(data))
    print("Participants:", data["Id"].nunique())
    print("Missing values:")
    print(data[["Id", "date", "logId"]].isna().sum())
    print("Duplicate rows:", data.duplicated().sum())


Sleep Export 3
Rows: 198559
Participants: 23
Missing values:
Id       0
date     0
logId    0
dtype: int64
Duplicate rows: 525

Sleep Export 4
Rows: 188521
Participants: 24
Missing values:
Id       0
date     0
logId    0
dtype: int64
Duplicate rows: 543


## Investigate Duplicate Records

Examine duplicate minute-level records within each sleep export and identify records shared between exports. Determine the sources of duplication before removing records that could inflate calculated sleep durations.

In [5]:
sleep3[sleep3.duplicated(keep=False)].sort_values(
    ['Id', 'date', 'logId']
).head(20)

,Id,date,value,logId
79892,4319703577,4/5/2016 10:50:00 PM,3,11344563687
80158,4319703577,4/5/2016 10:50:00 PM,3,11344563687
79893,4319703577,4/5/2016 10:51:00 PM,3,11344563687
80159,4319703577,4/5/2016 10:51:00 PM,3,11344563687
79894,4319703577,4/5/2016 10:52:00 PM,2,11344563687
80160,4319703577,4/5/2016 10:52:00 PM,2,11344563687
79895,4319703577,4/5/2016 10:53:00 PM,2,11344563687
80161,4319703577,4/5/2016 10:53:00 PM,2,11344563687
79896,4319703577,4/5/2016 10:54:00 PM,2,11344563687
80162,4319703577,4/5/2016 10:54:00 PM,2,11344563687


In [6]:
sleep3.loc[sleep3.duplicated(keep=False)].groupby('logId').agg(
    TotalRows=('Id', 'size'),
    UniqueMinutes=('date', 'nunique')
)

,TotalRows,UniqueMinutes
logId,,
11344563687,532,266
11344563688,518,259


In [7]:
sleep4.loc[sleep4.duplicated(keep=False)].groupby('logId').agg(
    TotalRows=('Id', 'size'),
    UniqueMinutes=('date', 'nunique')
)

,TotalRows,UniqueMinutes
logId,,
11573168523,1086,543


The duplicate records within the original exports were traced to three sleep logs containing repeated minute-level observations. Next, the two exports were compared to identify additional duplicate records resulting from overlapping collection periods.

In [8]:
sleep_overlap = pd.merge(
    sleep3.drop_duplicates(),
    sleep4.drop_duplicates(),
    on=['Id', 'date', 'value', 'logId'],
    how='inner'
)

print("Records appearing in both exports:", len(sleep_overlap))

Records appearing in both exports: 3232


In [9]:
overlap_dates = pd.to_datetime(
    sleep_overlap['date'],
    format='%m/%d/%Y %I:%M:%S %p'
)

print("Earliest overlapping record:", overlap_dates.min())
print("Latest overlapping record:", overlap_dates.max())

Earliest overlapping record: 2016-04-11 20:48:00
Latest overlapping record: 2016-04-12 08:35:00


The investigation identified 1,068 duplicate rows within three sleep logs across the original exports (525 in Sleep 3 and 543 in Sleep 4). An additional 3,232 identical records appeared in both exports, spanning April 11–12, 2016. 

## Combine and Clean Sleep Records

Combine the two sleep exports into a single dataset and remove all 4,300 duplicate rows to prevent repeated minute-level observations from inflating calculated sleep durations. Verify the resulting record count and number of unique participants.

In [10]:
sleep_clean_combined = pd.concat([sleep3,sleep4]).drop_duplicates()

In [11]:
print("Combined rows after deduplication:", len(sleep_clean_combined))
print("Unique participants:", sleep_clean_combined["Id"].nunique())

Combined rows after deduplication: 382780
Unique participants: 25


## Reconstruct Sleep Sessions

Convert the timestamps to datetime format, then group minute-level records by participant ID and sleep log ID to reconstruct individual sleep sessions.

For each session, calculate the start time, end time, and total recorded sleep minutes. Extract the calendar date for subsequent participant-day analysis.

In [12]:
sleep_clean_combined['date'] = pd.to_datetime(sleep_clean_combined['date'])

In [13]:
sleep_sessions = (sleep_clean_combined.groupby(['Id','logId']).agg(SleepStart=('date','min'),SleepEnd=('date','max'),SleepMinutes=('date','size')).reset_index())

In [14]:
sleep_sessions['Date'] = sleep_sessions['SleepStart'].dt.date

In [15]:
print("Total sleep sessions:", len(sleep_sessions))
print("Unique participants:", sleep_sessions["Id"].nunique())

Total sleep sessions: 1007
Unique participants: 25


In [16]:
sleep_sessions.head()

,Id,logId,SleepStart,SleepEnd,SleepMinutes,Date
0,1503960366,11114919637,2016-03-13 02:39:30,2016-03-13 09:44:30,426,2016-03-13
1,1503960366,11126343681,2016-03-14 01:32:00,2016-03-14 07:57:00,386,2016-03-14
2,1503960366,11134971215,2016-03-15 02:36:00,2016-03-15 08:10:00,335,2016-03-15
3,1503960366,11142197163,2016-03-16 03:12:00,2016-03-16 08:14:00,303,2016-03-16
4,1503960366,11142197164,2016-03-16 19:43:00,2016-03-16 20:45:00,63,2016-03-16


## Prepare Sleep Participant-Date Records

Select the participant ID and calendar date from the reconstructed sleep sessions, removing duplicate combinations so that each participant appears only once per day.

This dataset was used as the sleep-data input when constructing the participant-day footprint visualization in the Bellabeat capstone. The complete footprint also incorporated other Fitbit data types and was created separately.

In [17]:
sleep_combined_dates = sleep_sessions[['Id','Date']].drop_duplicates()

In [18]:
sleep_combined_dates.head()

,Id,Date
0,1503960366,2016-03-13
1,1503960366,2016-03-14
2,1503960366,2016-03-15
3,1503960366,2016-03-16
5,1503960366,2016-03-17


## Export Sleep Session Data

The cleaned and aggregated sleep session data is exported to Excel for use in creating the sleep duration distribution histogram included in the Bellabeat capstone presentation.

In [19]:
sleep_sessions.to_excel("sleep_sessions.xlsx", index=False)